In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from scipy.optimize import minimize
from scipy.optimize import root_scalar

# Code finds minimum value based on the subharmonic frequency prediction
# for varying viscosity


# Constants
g = 9.81
sigma = 0.072
rho = 1000
#nu = 1e-6
f_d = 20.0
omega_d = 2 * np.pi * f_d
h_depth = 0.005

a_vals = np.linspace(0, 200, 60)

def dispersion_relation(k):
    """Dispersion relation for gravity-capillary waves"""
    return np.sqrt(g * k + (sigma / rho) * k**3) * np.tanh(k * h_depth)

def inverse_dispersion_relation(omega, kmin=0, kmax=1e4):
    """Finds k from omega_d"""
    sol = root_scalar(
        lambda k: dispersion_relation(k) - omega,
        bracket=[kmin, kmax],
        method='brentq')

    return sol.root

def get_gamma_exact(k, nu):
    """Gamma factor according to a research paper"""
    return nu*k**2 * (2 + 1/(np.tanh(2*k*h_depth)*np.sinh(2*k*h_depth))) + np.sqrt(k*nu*np.sqrt(9.81*h_depth)/8)*2*k/np.sinh(2*k*h_depth)

def system_dynamics(y, t_scalar, k, A, nu):
    """Function guards the system dynamics and will be numerically solved"""
    h, v = y
    tanh_term = np.tanh(k * h_depth)
    omega0_sq = dispersion_relation(k)**2
    gamma = get_gamma_exact(k, nu)

    # The forcing modulates gravity specifically
    # forcing = (A * k * tanh(kh)) * cos(omega_d * t)
    forcing = (A * k * tanh_term) * np.cos(omega_d * t_scalar)

    # Dynamics of system
    dhdt = v
    dvdt = -2 * gamma * v - (omega0_sq - forcing) * h
    return [dhdt, dvdt]

def instability_measure(k, A, nu):
    """Returns real part of eigenvalues"""
    T = 2 * np.pi / omega_d
    t = [0, T]
    # Monodromy matrix construction
    res1 = odeint(system_dynamics, [1.0, 0.0], t, args=(k, A, nu))[-1]
    res2 = odeint(system_dynamics, [0.0, 1.0], t, args=(k, A, nu))[-1]
    M = np.column_stack([res1, res2])
    eigenvalues = np.linalg.eigvals(M)
    return np.real(max(eigenvalues, key=lambda x: abs(x)))

def boundary_finder(k_val, nu):
    """Function that finds acceleration boundary for given k"""
    if isinstance(k_val, np.ndarray):
        k_val = k_val.item() # Convert [val] to val

    val_2 = instability_measure(k_val, a_vals[0], nu) # Start cycle properly
    a_2 = a_vals[0]
    for a in a_vals[1:]:
        val_1 = val_2 # Optimization to prevent calculating the same thing twice
        val_2 = instability_measure(k_val, a, nu)
        a_1 = a_2
        a_2 = a

        # Checks boundary of stability with instability
        if (abs(val_2)-1)*(abs(val_1)-1) < 0:
            sol = root_scalar(lambda a: abs(instability_measure(k_val, a, nu)) - 1,
                              bracket=[a_1, a_2], method='brentq')
            return sol.root


k_guess = inverse_dispersion_relation(omega_d/2) # Needed in following function
def minimum_finder(nu):
    """Finds minimum acceleration for given driving frequency"""
    res = minimize(boundary_finder, k_guess, args=(nu))
    k_opt = res.x[0]
    f_opt = dispersion_relation(k_opt)/(2*np.pi)
    a_min = res.fun
    return a_min, k_opt, f_opt

# Loop over logspace
nu_vals = np.logspace(-6, -4, 20)
minima = []
a = []
k = []
f = []

for nu in nu_vals:
    a_res, k_res, f_res = minimum_finder(nu)
    a.append(a_res)
    k.append(k_res)
    f.append(f_res)

# Plotting and formatting
plt.plot(nu_vals, np.array(a)/g, color="green", linestyle="--", label="Numerical model")
plt.xscale("log")
plt.xlabel(r"Viscosity $\nu$ m$^{2}$/s)")
plt.ylabel("On-set Gamma $\Gamma_z$")
plt.legend()
plt.grid(linestyle="--", alpha=0.5)
plt.xticks(np.hstack((np.linspace(1e-6, 1e-5, 10), np.linspace(2e-5, 1e-4, 9))))
plt.show()